# lineup — model stages on a free cloud GPU

This notebook runs the GPU stages of the benchmark on a free Colab or Kaggle T4 (16 GB). It clones the repository, builds test cases on CPU, and loads Qwen2.5-7B-Instruct in 4-bit to run generation, the leave-one-out oracle, the attribution methods under test, and the scorer.

Before running, set the runtime to a GPU: **Runtime → Change runtime type → T4 GPU** (on Kaggle, enable the GPU accelerator and Internet in the sidebar).

In [ ]:
!git clone https://github.com/santoshcheethiralame-dot/LINEUP
%cd LINEUP
!pip install -q -e .
!pip install -q bitsandbytes

## Load Qwen2.5-7B in 4-bit

The first load downloads about 5 GB of 4-bit weights.

In [ ]:
from lineup.backends import TransformersModel
from lineup.config import DEFAULT_MODEL, set_seed

set_seed()
model = TransformersModel(DEFAULT_MODEL, load_in_4bit=True, max_new_tokens=64)
print("loaded", DEFAULT_MODEL)

## Build the cases and run the model

Scenario construction is model-free and deterministic; the same seed reproduces the same benchmark anywhere.

In [ ]:
from lineup.data.hotpotqa import load_examples
from lineup.data.substitution import build_answer_pool
from lineup.data.scenario import ScenarioBuilder
from lineup.data.misleading import substitution_check
from lineup.correctness import LLMJudge
from lineup.generation import generate_and_judge

examples = list(load_examples("validation", limit=50))
pool = build_answer_pool(examples)
builder = ScenarioBuilder(answer_pool=pool, seed=0)
judge = LLMJudge(model)

scenarios, originals = [], []
for example in examples:
    if substitution_check(example):
        continue
    scenario = builder.build(example)
    if scenario is None:
        continue
    scenarios.append(scenario)
    originals.append(generate_and_judge(model, scenario, llm_judge=judge))

wrong = [r for r in originals if not r.is_correct]
print(f"built {len(scenarios)} cases, {len(wrong)} answered wrongly")

## Stage 4 — counterfactual oracle

Leave-one-out over the wrong cases, accumulating the role labels for the scorer.

In [ ]:
from collections import Counter
from lineup.oracle import leave_one_out

role_cases = []
misleading_lands = Counter()
for scenario, original in zip(scenarios, originals):
    if original.is_correct:
        continue
    case = leave_one_out(model, scenario, original, llm_judge=judge)
    role_cases.append(case)
    for chunk_role in case.chunk_roles:
        if chunk_role.provenance == "misleading":
            misleading_lands[chunk_role.role] += 1

print("misleading chunk lands as:", dict(misleading_lands))

## Stage 5 — methods under test

Run each attribution method on the same wrong cases, accumulating predictions.

In [ ]:
from lineup.methods import ContextCite, LexicalSimilarity, LLMJudgeCulprit, run_method

methods = [ContextCite(n_ablations=16, seed=0), LexicalSimilarity(), LLMJudgeCulprit()]
predictions = []
for scenario, original in zip(scenarios, originals):
    if original.is_correct:
        continue
    for method in methods:
        predictions.append(run_method(method, model, scenario, original.model_answer))

print(f"{len(predictions)} predictions over {len(role_cases)} cases")

## Stage 6 — scorer

Grade each method against the oracle. A salience-based method should show a high `misleading-as-culprit` rate and a low `culprit > misleading` win-rate.

In [ ]:
from lineup.scoring import score_predictions

for report in score_predictions(role_cases, predictions):
    winrate = report.culprit_over_misleading_winrate
    winrate = f"{winrate:.2f}" if winrate is not None else "n/a"
    print(f"{report.method:18s}  top1={report.top1_culprit_accuracy:.2f}  "
          f"misleading-as-culprit={report.misleading_as_culprit_rate:.2f}  culprit>misleading={winrate}")

## Notes

- 4-bit (nf4) keeps the 7B model within a 16 GB T4; fp16 compute is used because the T4 has no native bfloat16.
- Run **all** model stages on one machine and one pinned model revision — greedy decoding is deterministic per machine, but log-probabilities drift across GPUs and precisions, so the labels must come from a single box.
- The scenarios themselves are machine-independent, so they can be rebuilt anywhere from the same seed.